In [ ]:
# Automation script needed in both files

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

sns.set_theme(style='darkgrid')

def evaluate_and_plot_pipeline(models_dict, X_test_dict, y_test_dict, target_names=None):
    """
    Automates metrics extraction and generates subplots for Confusion Matrices, Accuracy, and F1-score comparisons across Classical, Deep Learning, and Adversarial settings.
    
    Parameters:
    -----------
    models_dict : dict
        Format: { 'Model Name (e.g., PyTorch MLP - Clean)': model_object, ... }
    X_test_dict : dict
        Format: { 'Model Name': X_test_array_or_tensor, ... }
    y_test_dict : dict
        Format: { 'Model Name': y_test_true_labels, ... }
    """
    results = []
    confusion_matrices = {}
    
    # Here gather metrics & store confusion matrices
    for name, model in models_dict.items():
        X_data = X_test_dict[name]
        y_true = y_test_dict[name]
        
        # PyTorch model prediction logic
        if hasattr(model, 'eval') and not hasattr(model, 'predict'):
            import torch
            model.eval()
            with torch.no_grad():
                # Convert data if needed
                if not isinstance(X_data, torch.Tensor):
                    X_tensor = torch.tensor(X_data, dtype=torch.float32)
                else:
                    X_tensor = X_data
                
                # Push to device
                device = next(model.parameters()).device
                outputs = model(X_tensor.to(device))
                y_pred = torch.argmax(outputs, dim=1).cpu().numpy()
                
                if isinstance(y_true, torch.Tensor):
                    y_true = y_true.numpy()
        else:
            # Scikit-Learn model prediction logic
            y_pred = model.predict(X_data)
            
        # Compute metrics
        acc = accuracy_score(y_true, y_pred)
        f1_weighted = f1_score(y_true, y_pred, average='weighted')
        cm = confusion_matrix(y_true, y_pred)
        
        confusion_matrices[name] = cm
        results.append({
            'Model Configuration': name,
            'Accuracy': acc,
            'Weighted F1-Score': f1_weighted
        })
        
        print(f"=== Classification Report: {name} ===")
        print(classification_report(y_true, y_pred, target_names=target_names))
        
    df_results = pd.DataFrame(results)
    
    # Plotting Setup
    num_models = len(models_dict)
    
    # If more than 2 models, use 3-row layout bc otherwise formats strangely
    if num_models > 2:
        fig = plt.figure(figsize=(16, 4 * num_models))
        gs = fig.add_gridspec(3, 2)
        
        # Row 1 & 2: Confusion Matrices (2x2 grids)
        for idx, (name, cm) in enumerate(confusion_matrices.items()):
            row = idx // 2
            col = idx % 2
            ax = fig.add_subplot(gs[row, col])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
            ax.set_title(f'{name}\nConfusion Matrix', fontsize=12, fontweight='bold')
            ax.set_xlabel('Predicted Label')
            ax.set_ylabel('True Label')
            
        # Row 3: Performance Comparisons (across columns)
        ax_acc = fig.add_subplot(gs[2, 0])
        ax_f1 = fig.add_subplot(gs[2, 1])
    else:
        # For 2-model baselines - default layout
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        for idx, (name, cm) in enumerate(confusion_matrices.items()):
            ax = axes[0, idx]
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
            ax.set_title(f'{name}\nConfusion Matrix', fontweight='bold')
            ax.set_xlabel('Predicted Label')
            ax.set_ylabel('True Label')
        ax_acc = axes[1, 0]
        ax_f1 = axes[1, 1]

    # Barplots
    sns.barplot(x='Accuracy', y='Model Configuration', data=df_results, palette='Blues_r', ax=ax_acc)
    ax_acc.set_xlim([0, 1.1])
    ax_acc.set_title('Accuracy Performance Comparison', fontsize=12, fontweight='bold')
    for p in ax_acc.patches:
        ax_acc.annotate(f'{p.get_width():.3f}', (p.get_width() + 0.01, p.get_y() + p.get_height()/2), va='center')
        
    sns.barplot(x='Weighted F1-Score', y='Model Configuration', data=df_results, palette='Greens_r', ax=ax_f1)
    ax_f1.set_xlim([0, 1.1])
    ax_f1.set_title('Weighted F1-Score Comparison', fontsize=12, fontweight='bold')
    ax_f1.set_ylabel('') 
    for p in ax_f1.patches:
        ax_f1.annotate(f'{p.get_width():.3f}', (p.get_width() + 0.01, p.get_y() + p.get_height()/2), va='center')
        
    plt.tight_layout()
    plt.show()
    return df_results

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

csv_path = "/kaggle/input/datasets/ericanacletoribeiro/cicids2017-cleaned-and-preprocessed/cicids2017_cleaned.csv"

df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)

df.head()

In [ ]:
print(df.columns[-5:])

In [ ]:
# Separate features and label
X = df.drop(columns=['Attack Type'])
y = df['Attack Type']

print("X shape:", X.shape)
print("y shape:", y.shape)

y.unique()[:10]  # see some label values

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Number of classes:", len(le.classes_))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

# Define MLP
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_train.shape[1]   # 52
num_classes = len(le.classes_) # 7

mlp = MLP(input_dim, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)

print(mlp)

In [ ]:
epochs = 5

for epoch in range(epochs):
    mlp.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = mlp(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
mlp.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)

        outputs = mlp(batch_X)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(batch_y.numpy())

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

print("Clean MLP Accuracy:", accuracy)
print("Clean MLP Weighted F1:", f1)
print(classification_report(all_labels, all_preds, target_names=le.classes_))

In [ ]:
def fgsm_attack(model, X, y, epsilon):
    X_adv = X.clone().detach().to(device)
    y = y.to(device)

    X_adv.requires_grad = True

    outputs = model(X_adv)
    loss = criterion(outputs, y)

    model.zero_grad()
    loss.backward()

    # FGSM perturbation
    perturbation = epsilon * X_adv.grad.sign()
    X_adv = X_adv + perturbation

    return X_adv.detach()

In [ ]:
epsilon = 0.05

mlp.eval()
fgsm_preds = []
fgsm_labels = []

adv_batches = []
label_batches = []

for batch_X, batch_y in test_loader:
    batch_X = batch_X.to(device)
    batch_y = batch_y.to(device)

    adv_X = fgsm_attack(mlp, batch_X, batch_y, epsilon)

    # save full adversarial dataset
    adv_batches.append(adv_X.cpu())
    label_batches.append(batch_y.cpu())

    # evaluate MLP on adversarial data
    with torch.no_grad():
        outputs = mlp(adv_X)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

    fgsm_preds.extend(preds)
    fgsm_labels.extend(batch_y.cpu().numpy())

X_test_fgsm = torch.cat(adv_batches).numpy()
y_test_fgsm = torch.cat(label_batches).numpy()

fgsm_acc = accuracy_score(fgsm_labels, fgsm_preds)
fgsm_f1 = f1_score(fgsm_labels, fgsm_preds, average="weighted")

print("FGSM Epsilon:", epsilon)
print("X_test_fgsm shape:", X_test_fgsm.shape)
print("FGSM Accuracy:", fgsm_acc)
print("FGSM Weighted F1:", fgsm_f1)
print("Accuracy Drop:", accuracy - fgsm_acc)
print("F1 Drop:", f1 - fgsm_f1)
print(classification_report(fgsm_labels, fgsm_preds, target_names=le.classes_))

In [ ]:
def pgd_attack(model, X, y, epsilon=0.05, alpha=0.01, num_iter=10):
    X_original = X.clone().detach().to(device)
    y = y.to(device)

    # Start from the original input
    X_adv = X_original.clone().detach()

    for _ in range(num_iter):
        X_adv.requires_grad = True

        outputs = model(X_adv)
        loss = criterion(outputs, y)

        model.zero_grad()
        loss.backward()

        # Small FGSM-like step
        X_adv = X_adv + alpha * X_adv.grad.sign()

        # Project back into epsilon-ball around original input
        perturbation = torch.clamp(X_adv - X_original, min=-epsilon, max=epsilon)
        X_adv = torch.clamp(X_original + perturbation, min=X_train.min(), max=X_train.max()).detach()

    return X_adv

In [ ]:
epsilon = 0.05
alpha = 0.01
num_iter = 10

mlp.eval()
pgd_preds = []
pgd_labels = []

pgd_batches = []
pgd_label_batches = []

for batch_X, batch_y in test_loader:
    batch_X = batch_X.to(device)
    batch_y = batch_y.to(device)

    adv_X = pgd_attack(mlp, batch_X, batch_y, epsilon, alpha, num_iter)

    pgd_batches.append(adv_X.cpu())
    pgd_label_batches.append(batch_y.cpu())

    with torch.no_grad():
        outputs = mlp(adv_X)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

    pgd_preds.extend(preds)
    pgd_labels.extend(batch_y.cpu().numpy())

X_test_pgd = torch.cat(pgd_batches).numpy()
y_test_pgd = torch.cat(pgd_label_batches).numpy()

pgd_acc = accuracy_score(pgd_labels, pgd_preds)
pgd_f1 = f1_score(pgd_labels, pgd_preds, average="weighted")

print("PGD Epsilon:", epsilon)
print("PGD Alpha:", alpha)
print("PGD Iterations:", num_iter)
print("X_test_pgd shape:", X_test_pgd.shape)
print("PGD Accuracy:", pgd_acc)
print("PGD Weighted F1:", pgd_f1)
print("Accuracy Drop:", accuracy - pgd_acc)
print("F1 Drop:", f1 - pgd_f1)

In [ ]:
torch.save(mlp.state_dict(), "/kaggle/working/mlp_model.pth")

In [ ]:
import json

results = {
    "clean_accuracy": accuracy,
    "clean_f1": f1,
    "fgsm_accuracy": fgsm_acc,
    "fgsm_f1": fgsm_f1,
    "pgd_accuracy": pgd_acc,
    "pgd_f1": pgd_f1
}

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# Use smaller training subset so it runs faster
N = 300000
X_train_small = X_train[:N]
y_train_small = y_train[:N]

lr_model = LogisticRegression(max_iter=500, n_jobs=-1)
lr_model.fit(X_train_small, y_train_small)

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_small, y_train_small)

for name, model in [("LR", lr_model), ("RF", rf_model)]:
    preds = model.predict(X_test)
    print(name, "Clean Accuracy:", accuracy_score(y_test, preds))
    print(name, "Clean F1:", f1_score(y_test, preds, average="weighted"))

In [ ]:
print(X_test_fgsm.shape)
print(X_test.shape)

In [ ]:
for name, model in [("LR", lr_model), ("RF", rf_model)]:
    preds = model.predict(X_test_fgsm)
    print(name, "FGSM Accuracy:", accuracy_score(y_test_fgsm, preds))
    print(name, "FGSM F1:", f1_score(y_test_fgsm, preds, average="weighted"))

In [ ]:
print(X_test_pgd.shape)
print(X_test.shape)

In [ ]:
for name, model in [("LR", lr_model), ("RF", rf_model)]:
    preds = model.predict(X_test_pgd)
    print(name, "PGD Accuracy:", accuracy_score(y_test_pgd, preds))
    print(name, "PGD F1:", f1_score(y_test_pgd, preds, average="weighted"))

In [ ]:
# Automated models for this file -- the adversarial example result metrics

models = {
    "MLP Clean Data": mlp,
    "MLP under FGSM Attack": mlp,
    "MLP under PGD Attack": mlp,
    "Random Forest Transfer (FGSM)": rf_model
}

X_tests = {
    "MLP Clean Data": X_test_tensor,
    "MLP under FGSM Attack": torch.tensor(X_test_fgsm),
    "MLP under PGD Attack": torch.tensor(X_test_pgd),
    "Random Forest Transfer (FGSM)": X_test_fgsm
}

y_tests = {
    "MLP Clean Data": y_test_tensor,
    "MLP under FGSM Attack": torch.tensor(y_test_fgsm),
    "MLP under PGD Attack": torch.tensor(y_test_pgd),
    "Random Forest Transfer (FGSM)": y_test_fgsm
}

df_summary = evaluate_and_plot_pipeline(models, X_tests, y_tests, target_names=le.classes_)